# 03 — Regressão: quanto tempo o voo vai atrasar?

**Tarefa:** prever `ARRIVAL_DELAY` em minutos (regressão).

**Algoritmos comparados (≥2):**
1. **Ridge Regression** (baseline linear regularizado).
2. **LightGBM Regressor** (gradient boosting em árvores).

**Métricas:** MAE, RMSE, R².

**Cuidado com a cauda longa:** a distribuição de `ARRIVAL_DELAY` é fortemente assimétrica (poucos voos com 200+ min puxam a média). Aplicamos `signed log1p` no alvo para estabilizar a variância.

In [ ]:
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

from src import config
from src.data_loader import load_flights
from src.preprocessing import clean_pipeline
from src.features import feature_pipeline
from src.models.regression import (
    build_feature_matrix,
    build_ridge_pipeline,
    build_lightgbm_regressor,
)
from src.evaluation import metrics_table, plot_residuals, regression_metrics

sns.set_theme(style="whitegrid")

## 1. Preparar dados

In [ ]:
df = load_flights(sample_size=config.SAMPLE_SIZE)
df = clean_pipeline(df)
df = feature_pipeline(df)
print(f"Shape: {df.shape}")
print(df["ARRIVAL_DELAY"].describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(df["ARRIVAL_DELAY"].clip(-60, 180), bins=80, color="steelblue", edgecolor="white")
axes[0].set_title("ARRIVAL_DELAY (clipped –60..180)")
axes[0].set_xlabel("min")
log_y = np.sign(df["ARRIVAL_DELAY"]) * np.log1p(np.abs(df["ARRIVAL_DELAY"]))
axes[1].hist(log_y, bins=80, color="darkorange", edgecolor="white")
axes[1].set_title("signed log1p(ARRIVAL_DELAY) — alvo transformado")
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "11_target_distribution_reg.png", dpi=120)
plt.show()

In [ ]:
X = build_feature_matrix(df)
y = df["ARRIVAL_DELAY"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=config.RANDOM_SEED
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 2. Modelo 1 — Ridge Regression

Linear, com regularização L2. O alvo é envelopado em log1p (`TransformedTargetRegressor`).

In [ ]:
ridge = build_ridge_pipeline(config.RANDOM_SEED, alpha=1.0, log_target=True)
t0 = time.time()
ridge.fit(X_train, y_train)
print(f"Treinamento: {time.time() - t0:.1f}s")
y_pred_ridge = ridge.predict(X_test)
metrics_ridge = regression_metrics(y_test, y_pred_ridge)
metrics_ridge

## 3. Modelo 2 — LightGBM Regressor

In [ ]:
lgbm = build_lightgbm_regressor(
    config.RANDOM_SEED, n_estimators=500, learning_rate=0.05, log_target=True
)
t0 = time.time()
lgbm.fit(X_train, y_train)
print(f"Treinamento: {time.time() - t0:.1f}s")
y_pred_lgbm = lgbm.predict(X_test)
metrics_lgbm = regression_metrics(y_test, y_pred_lgbm)
metrics_lgbm

## 4. Comparação

In [ ]:
summary = metrics_table({
    "Ridge": metrics_ridge,
    "LightGBM": metrics_lgbm,
})
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_residuals(y_test, y_pred_ridge, ax=axes[0], title="Ridge — resíduos")
plot_residuals(y_test, y_pred_lgbm, ax=axes[1], title="LightGBM — resíduos")
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "12_residuals.png", dpi=120)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
sample_idx = np.random.RandomState(config.RANDOM_SEED).choice(len(y_test), size=min(5000, len(y_test)), replace=False)
ax.scatter(y_test[sample_idx], y_pred_lgbm[sample_idx], alpha=0.2, s=8)
lo, hi = -50, 200
ax.plot([lo, hi], [lo, hi], "r--", alpha=0.6)
ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)
ax.set_xlabel("Real (min)")
ax.set_ylabel("Predito (min)")
ax.set_title("LightGBM: predito × real (amostra)")
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "13_pred_vs_actual.png", dpi=120)
plt.show()

## 5. MAE por bucket de magnitude

O modelo é melhor onde? Vamos quebrar o MAE em buckets do atraso real.

In [ ]:
df_eval = pd.DataFrame({"real": y_test, "pred": y_pred_lgbm})
df_eval["abs_err"] = (df_eval["real"] - df_eval["pred"]).abs()
df_eval["bucket"] = pd.cut(
    df_eval["real"],
    bins=[-np.inf, -10, 0, 15, 30, 60, 120, np.inf],
    labels=["<-10", "-10..0", "0..15", "15..30", "30..60", "60..120", ">120"],
)
bucket_mae = df_eval.groupby("bucket", observed=True)["abs_err"].agg(["mean", "count"]).round(2)
bucket_mae

## 6. Persistir o melhor modelo

In [ ]:
best = lgbm
out_path = config.MODELS_DIR / "reg_lightgbm_best.joblib"
joblib.dump(best, out_path)
print(f"Modelo salvo em {out_path}")
summary.to_csv(config.REPORTS_DIR / "metrics_regression.csv")

## 7. Discussão

**Conclusões esperadas**:
- O MAE global tende a ficar em ~14–18 min sem `DEPARTURE_DELAY` — o problema sem informações operacionais é bem ruidoso.
- O modelo acerta o cenário comum (voos próximos ao horário) e erra mais nas caudas (atrasos extremos).
- LightGBM > Ridge por margem clara, especialmente em RMSE (cauda longa).

**Limitações**:
- Atrasos extremos (>120 min) são raros e quase aleatórios — sem clima/operações em tempo real, é impossível prever bem.
- A relação entre features (hora, companhia, aeroporto) e atraso é não-linear e cheia de interações, o que justifica o ganho de árvores.

**Próximos passos**:
- Modelar separadamente: (i) probabilidade de atrasar (classificação) e (ii) magnitude do atraso *dado que atrasou* (regressão condicional) — abordagem two-stage.
- Quantile regression para estimar intervalos de confiança (P50, P90).